In [1]:
!pip install --upgrade --no-cache-dir pykrx
from pykrx import stock
from datetime import date
import time
import pandas as pd
import numpy as np
import requests

You should consider upgrading via the '/Users/mingeonsung/esg-investment/venv/bin/python3 -m pip install --upgrade pip' command.
KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.


In [3]:
# 날짜 설정
today = date.today().strftime("%Y%m%d")
start_date = "20230101"  # 3년치 확보 (252일 룰 적용에 충분)
end_date = today

print(f"Data range: {start_date} ~ {end_date}")

tickers = ["069500","229200","132030","219480","379810","192090","195980","251350",
           "352560","195930","245360","314250","289250","290130","379800","228790",
           "462010","371160","466920","457480","305720","371460","449450","494670",
           "487240","458730","483320","381180","487230","498400","091160"]

price = pd.DataFrame()
print("데이터 다운로드 시작...")
for ticker in tickers:
    df = stock.get_market_ohlcv_by_date(start_date, end_date, ticker)
    if not df.empty:
        price = pd.concat([price, df["종가"].rename(ticker)], axis=1)
        print(f"{ticker} 완료")
    else:
        print(f"데이터 없음: {ticker}")
    time.sleep(0.05)

price.to_csv("price.csv", encoding="utf-8-sig")
print(f"price.csv 저장! ({price.shape[0]}일 x {price.shape[1]}개 ETF)")
display(price.tail())

Data range: 20230101 ~ 20260602
데이터 다운로드 시작...
069500 완료
229200 완료
132030 완료
219480 완료
379810 완료
192090 완료
195980 완료
251350 완료
352560 완료
195930 완료
245360 완료
314250 완료
289250 완료
290130 완료
379800 완료
228790 완료
462010 완료
371160 완료
466920 완료
457480 완료
305720 완료
371460 완료
449450 완료
494670 완료
487240 완료
458730 완료
483320 완료
381180 완료
487230 완료
498400 완료
091160 완료
price.csv 저장! (832일 x 31개 ETF)


,069500,229200,132030,219480,379810,192090,195980,251350,352560,195930,...,371460,449450,494670,487240,458730,483320,381180,487230,498400,091160
2026-05-27 00:00:00,130600,19465,25390,32610,29865,14335,15400,41515,12540,22275,...,13850,71865.0,31850.0,51110.0,15600.0,28480.0,50735,26395.0,25650.0,163570
2026-05-28 00:00:00,129990,19120,24790,32525,29760,14315,15195,41320,12450,22005,...,13755,70125.0,30125.0,48565.0,15535.0,28325.0,49355,25800.0,25655.0,160810
2026-05-29 00:00:00,134815,18730,25535,32805,30250,14400,15480,41925,12365,22125,...,13700,68130.0,29780.0,47220.0,15540.0,29225.0,50825,25890.0,26545.0,160980
2026-06-01 00:00:00,141395,18410,25530,32920,30435,14205,15705,42020,12280,22190,...,13500,67995.0,29470.0,50170.0,15450.0,29645.0,51220,25605.0,27800.0,165410
2026-06-02 00:00:00,141480,18235,25380,32780,30475,14175,15675,41970,12000,22175,...,13455,64090.0,28135.0,47540.0,15395.0,30625.0,51057,25155.0,27770.0,165815


In [4]:
import pandas as pd
import requests

# 1. 티커-종목명 매핑 생성
def fetch_etf_name(ticker):
    try:
        url = f'https://m.stock.naver.com/api/stock/{ticker}/basic'
        resp = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=5)
        if resp.ok:
            return resp.json().get('stockName', ticker)
    except Exception:
        pass
    return ticker

ticker_map = {ticker: fetch_etf_name(ticker) for ticker in price.columns}

# 2. 변환 함수
def get_name(ticker):
    return ticker_map.get(ticker, ticker) # 매핑에 없는 경우 티커 자체 반환

# price.csv 파일 로드 (이전 셀에서 생성된 파일)
price = pd.read_csv('price.csv', index_col=0)

# 컬럼명(티커 코드)을 종목명으로 변환
price.columns = [get_name(col) for col in price.columns]

# 변환된 DataFrame 상위 5행 표시
display(price.head())

,KODEX 200,KODEX 코스닥150,KODEX 골드선물(H),KODEX 미국S&P500선물(H),KODEX 미국나스닥100,TIGER 차이나CSI300,PLUS 신흥국MSCI(합성 H),KODEX MSCI선진국,KODEX 미국부동산리츠(H),TIGER 유로스탁스50(합성 H),...,TIGER 차이나전기차SOLACTIVE,PLUS K방산,TIGER 조선TOP10,KODEX AI전력핵심설비,TIGER 미국배당다우존스,ACE 엔비디아밸류체인액티브,TIGER 미국필라델피아반도체나스닥,KODEX 미국AI전력핵심인프라,KODEX 200타겟위클리커버드콜,KODEX 반도체
2023-01-02 00:00:00,27513,9479,12159,18168,9070,8929,8961,18056,10179,13169,...,11536,NaN,NaN,NaN,NaN,NaN,8261,NaN,NaN,21366
2023-01-03 00:00:00,27513,9524,12099,18232,9133,9039,8911,18406,10246,13339,...,11679,NaN,NaN,NaN,NaN,NaN,8425,NaN,NaN,21533
2023-01-04 00:00:00,28093,9584,12139,18113,9070,9044,8966,18330,10241,13494,...,11343,NaN,NaN,NaN,NaN,NaN,8348,NaN,NaN,22631
2023-01-05 00:00:00,28317,9470,12169,18197,9070,9259,9166,18411,10426,13793,...,11649,9091.0,NaN,NaN,NaN,NaN,8480,NaN,NaN,22471
2023-01-06 00:00:00,28619,9668,12065,18054,8951,9303,9101,18267,10156,13823,...,11902,9136.0,NaN,NaN,NaN,NaN,8378,NaN,NaN,23110


In [5]:
MIN_HISTORY = 252

def get_universe_ranks(price_data, rebal_date):
    """Dynamic Universe Rule: only ETFs with >= 252 valid trading days as of rebal_date."""
    data = price_data[price_data.index <= str(rebal_date)]

    valid_counts = data.notna().sum()
    universe = valid_counts[valid_counts >= MIN_HISTORY].index.tolist()
    excluded  = valid_counts[valid_counts <  MIN_HISTORY]

    print(f"Rebalancing: {str(rebal_date)[:10]}  |  Universe: {len(universe)}/{len(price_data.columns)} ETFs")
    for etf, cnt in excluded.items():
        print(f"  Excluded {etf}: only {int(cnt)} trading days")

    results = {}
    for etf in universe:
        valid   = data[etf].dropna()                # never use pre-listing data
        base    = valid.iloc[-MIN_HISTORY]           # price 252 valid days ago
        last    = valid.iloc[-1]
        log_ret = np.log(valid.tail(MIN_HISTORY) / valid.tail(MIN_HISTORY).shift(1)).dropna()
        results[etf] = {
            "momentum": round(last / base, 3),
            "vol":      log_ret.std() * np.sqrt(250),
        }

    df = pd.DataFrame(results).T
    df["mrank"]  = df["momentum"].rank(ascending=False, method="min")
    df["vrank"]  = df["vol"].rank(ascending=True,  method="min")
    n = len(df)
    df_sorted = df.sort_values("mrank").copy()
    df_sorted["mrank3"] = [(n - i) / 100 for i in range(n)]
    df["mrank3"] = df_sorted["mrank3"]
    df["frank"]  = df["vrank"] + df["mrank"] - df["mrank3"]
    return df.sort_values("frank")

# Run at last month-end rebalancing date
last_month_end = pd.Timestamp(price.index[-1]).to_period("M").to_timestamp() - pd.offsets.BDay(1)
rebal_date = price.index[price.index <= str(last_month_end)][-1]

rank2 = get_universe_ranks(price, rebal_date)
display(rank2[["momentum","vol","mrank","vrank","frank"]])

Rebalancing: 2026-05-29  |  Universe: 31/31 ETFs


,momentum,vol,mrank,vrank,frank
KODEX 미국S&P500,1.391,0.119032,21.0,1.0,21.89
ACE 엔비디아밸류체인액티브,3.543,0.364828,6.0,17.0,22.74
TIGER 미국필라델피아반도체나스닥,2.882,0.311624,8.0,15.0,22.76
TIGER 차이나CSI300,1.488,0.143772,18.0,5.0,22.87
KODEX 200,3.907,0.393474,5.0,19.0,23.73
KODEX 미국나스닥100,1.545,0.151629,17.0,7.0,23.85
KODEX MSCI선진국,1.366,0.119522,22.0,2.0,23.90
KODEX 200타겟위클리커버드콜,3.251,0.368663,7.0,18.0,24.75
RISE ESG사회책임투자,4.685,0.434402,2.0,24.0,25.70
TIGER MSCI KOREA ESG유니버설,4.248,0.408161,3.0,23.0,25.71


In [6]:
rank2

,momentum,vol,mrank,vrank,mrank3,frank
KODEX 미국S&P500,1.391,0.119032,21.0,1.0,0.11,21.89
ACE 엔비디아밸류체인액티브,3.543,0.364828,6.0,17.0,0.26,22.74
TIGER 미국필라델피아반도체나스닥,2.882,0.311624,8.0,15.0,0.24,22.76
TIGER 차이나CSI300,1.488,0.143772,18.0,5.0,0.13,22.87
KODEX 200,3.907,0.393474,5.0,19.0,0.27,23.73
KODEX 미국나스닥100,1.545,0.151629,17.0,7.0,0.15,23.85
KODEX MSCI선진국,1.366,0.119522,22.0,2.0,0.10,23.90
KODEX 200타겟위클리커버드콜,3.251,0.368663,7.0,18.0,0.25,24.75
RISE ESG사회책임투자,4.685,0.434402,2.0,24.0,0.30,25.70
TIGER MSCI KOREA ESG유니버설,4.248,0.408161,3.0,23.0,0.29,25.71


In [7]:
rank2.to_csv("rankings.csv", encoding="utf-8-sig")
print("rankings.csv 저장 완료!")

rankings.csv 저장 완료!


In [8]:
print("=== Top 10 ETFs ===")
display(rank2[["momentum","vol","mrank","vrank","frank"]].head(10))

=== Top 10 ETFs ===


,momentum,vol,mrank,vrank,frank
KODEX 미국S&P500,1.391,0.119032,21.0,1.0,21.89
ACE 엔비디아밸류체인액티브,3.543,0.364828,6.0,17.0,22.74
TIGER 미국필라델피아반도체나스닥,2.882,0.311624,8.0,15.0,22.76
TIGER 차이나CSI300,1.488,0.143772,18.0,5.0,22.87
KODEX 200,3.907,0.393474,5.0,19.0,23.73
KODEX 미국나스닥100,1.545,0.151629,17.0,7.0,23.85
KODEX MSCI선진국,1.366,0.119522,22.0,2.0,23.90
KODEX 200타겟위클리커버드콜,3.251,0.368663,7.0,18.0,24.75
RISE ESG사회책임투자,4.685,0.434402,2.0,24.0,25.70
TIGER MSCI KOREA ESG유니버설,4.248,0.408161,3.0,23.0,25.71


In [ ]:
ym1 = rank2.sort_values("mrank")[["momentum","mrank","vol","vrank","frank"]]
display(ym1)

In [ ]:
# 과거 리밸런싱 날짜로 조회 예시:
# result = get_universe_ranks(price, "2025-06-30")
# display(result)